# Phase 4 — Train the detector

**Runtime → Change runtime type → T4 GPU** before running anything here.

`torch` has no wheel for Python 3.13 on Intel macOS, so this cannot run on the
development machine at all. Colab is the only place it executes.

**Kill criterion:** mAP50 < 0.40 after full training → back to Phase 3.


In [ ]:
# Clone (or pull) and install. Re-run this cell after every `git push`.
import os, pathlib
if pathlib.Path('smugglers').exists():
    !cd smugglers && git pull --ff-only
else:
    !git clone https://github.com/Jaswanth-K1210/smugglers.git
%cd /content/smugglers
!pip install -q -r requirements.txt
print()
!git log --oneline -1


In [ ]:
# Credentials. Only GFW needs one — SAR and AIS are open.
# Colab: use the key icon in the left sidebar, add GFW_API_TOKEN, then run this.
from google.colab import userdata
import pathlib
tok = ""
try:
    tok = userdata.get('GFW_API_TOKEN')
except Exception:
    print("No Colab secret found. Paste your token below if you want GFW layers.")
pathlib.Path('.env').write_text(
    f"GFW_API_TOKEN={tok}\nAOI_BBOX=9.5,56.5,12.5,58.5\n")
from src.config import GFW_API_TOKEN, bbox
print("GFW token set:", bool(GFW_API_TOKEN), "| AOI:", bbox())


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
if not torch.cuda.is_available():
    print('\n!! CPU runtime. Runtime -> Change runtime type -> T4 GPU, then re-run.')


## 1. Restore tiles from Drive (if this is a fresh runtime)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r /content/drive/MyDrive/darksts/tiles data/ 2>/dev/null || echo "no Drive copy; run 03_autolabel first"
!ls data/tiles/images | wc -l


## 2. Build the dataset

Split is **by scene, never by tile**. Tiles from one scene share sea state and
often the same vessel minutes apart; a random tile split puts near-duplicates on
both sides and reports a score the model has not earned.

If it warns about a single scene, go back and label more — any mAP from one
scene is meaningless.

In [ ]:
from src.train import build_dataset
yml = build_dataset()
!cat {yml}


## 3. Which detector families does this build support?

In [ ]:
from src.train import available
available()


## 4. Three-epoch smoke test first

Confirm the loss falls before spending an hour. Stop here and report back.

In [ ]:
from src.train import train
train(yml, model='yolov8n.pt', epochs=3, batch=8, name='smoke', device=0)


## 5. Full run

In [ ]:
best = train(yml, model='yolov8n.pt', epochs=50, batch=8, name='darksts', device=0)
print(best)


## 6. Benchmark the detector family

Same split, same schedule — only the backbone varies, so the numbers are
comparable. This is the table the paper reports instead of asserting a winner.

Only worth running once you have tens of scenes; over 20 tiles it measures
noise.

In [ ]:
from src.train import benchmark
benchmark(['yolov8n.pt', 'yolo11n.pt', 'yolo12n.pt'], epochs=50, batch=8, device=0)


## 7. Validate and keep the weights

In [ ]:
from ultralytics import YOLO
m = YOLO(str(best))
r = m.val(data=str(yml), imgsz=1024, device=0)
print(f"mAP50 {r.box.map50:.4f}  mAP50-95 {r.box.map:.4f}  P {r.box.mp:.4f}  R {r.box.mr:.4f}")
print("KILL CRITERION:", "PASS" if r.box.map50 >= 0.40 else "FAIL -> back to Phase 3")


In [ ]:
!mkdir -p /content/drive/MyDrive/darksts/models
!cp {best} /content/drive/MyDrive/darksts/models/best.pt
!ls -la /content/drive/MyDrive/darksts/models/


---
**Report back:** the mAP50 line and the PASS/FAIL. On any error paste the
**entire traceback**, not the last line.